In [1]:
import os
from d3rlpy.logging import UnifiedFileAdapterFactory
import numpy as np
local = True
if local:
    os.environ["D3RLPY_DATASETS_PATH"] = "/home/julian/programming_data/d3rlpy_data"
    os.environ["MINARI_DATASETS_PATH"] = "/home/julian/programming_data/d3rlpy_data/minari_data"
else:
    # on cluster
    os.environ["D3RLPY_DATASETS_PATH"] = "/gpfs/data/fs72297/jklotz/programming_data/d3rlpy_data"
    os.environ["MINARI_DATASETS_PATH"] = "/gpfs/data/fs72297/jklotz/programming_data/d3rlpy_data/minari_data"
os.makedirs(os.environ["D3RLPY_DATASETS_PATH"], exist_ok=True)
os.makedirs(os.environ["MINARI_DATASETS_PATH"], exist_ok=True)

import d3rlpy

dataset, env = d3rlpy.datasets.get_minari('atari/pong/expert-v0')
seed = 1



dataset_name = "pong"


# settings in original DT paper
# Batch size 512 Pong 128 Breakout, Qbert, Seaquest Context length K 50 Pong 30 Breakout, Qbert, Seaquest Return-to-go conditioning 90 Breakout (≈ 1× max in dataset) 2500 Qbert (≈ 5× max in dataset) 20 Pong (≈ 1× max in dataset) 1450 Seaquest (≈ 5× max in dataset)

if dataset_name == "pong":
    batch_size = 512
    context_size = 50
    target_return = 20
elif dataset_name == "breakout":
    batch_size = 128
    context_size = 30
    target_return = 90
elif dataset_name == "qbert":
    batch_size = 128
    context_size = 30
    target_return = 2500
elif dataset_name == "seaquest":    
    batch_size = 128
    context_size = 30
    target_return = 1450

# fix seed
d3rlpy.seed(seed)
d3rlpy.envs.seed_env(env, seed)
target_return=10

discrete_tacr = d3rlpy.algos.DiscreteTACRConfig(
    batch_size=batch_size,
    actor_learning_rate=1e-4,
    actor_optim_factory=d3rlpy.optimizers.AdamWFactory(
        weight_decay=1e-4,
        clip_grad_norm=0.25,
        lr_scheduler_factory=d3rlpy.optimizers.WarmupSchedulerFactory(
            warmup_steps=10000#10000
        ),
    ),
    actor_encoder_factory=d3rlpy.models.PixelEncoderFactory(
            feature_size=128, exclude_last_activation=True
        ),
    observation_scaler=d3rlpy.preprocessing.PixelObservationScaler(),
    position_encoding_type=d3rlpy.PositionEncodingType.GLOBAL,
    context_size=context_size,
    num_heads=8,
    num_layers=6,
    max_timestep=2000,
    compile_graph=True,
    alpha=0.5,
).create(device="cpu")

discrete_tacr.fit(
    dataset,
    n_steps=100,# 100000,
    n_steps_per_epoch=10,# 1000,
    #save_interval=1,
    eval_env=env,
    eval_target_return=target_return,
    experiment_name=f"Discrete_TACR_{dataset_name}_{seed}",
    logger_adapter=UnifiedFileAdapterFactory(),
    n_trials=50,
    eval_gaps=1
)

/home/julian/miniconda3/envs/d3rlpy_dev_requirements_py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]


2025-07-28 21:35.43 [info     ] Signatures have been automatically determined. action_signature=Signature(dtype=[dtype('int64')], shape=[(1,)]) observation_signature=Signature(dtype=[dtype('uint8')], shape=[(3, 210, 160)]) reward_signature=Signature(dtype=[dtype('float64')], shape=[(1,)])
2025-07-28 21:35.43 [info     ] Action-space has been automatically determined. action_space=<ActionSpace.DISCRETE: 2>
2025-07-28 21:35.43 [info     ] Action size has been automatically determined. action_size=6
2025-07-28 21:35.44 [info     ] dataset info                   dataset_info=DatasetInfo(observation_signature=Signature(dtype=[dtype('uint8')], shape=[(3, 210, 160)]), action_signature=Signature(dtype=[dtype('int64')], shape=[(1,)]), reward_signature=Signature(dtype=[dtype('float64')], shape=[(1,)]), action_space=<ActionSpace.DISCRETE: 2>, action_size=6)
2025-07-28 21:35.44 [debug    ] Building models...            
2025-07-28 21:35.50 [debug    ] Models have been built.       
2025-07-28 21:3

Epoch 1/10:   0%|          | 0/10 [00:00<?, ?it/s]

: 

In [3]:
print("Max timestep in dataset:", max([len(ep.actions) for ep in dataset.episodes]))

Max timestep in dataset: 1962


In [6]:
type(env.reset())

tuple

In [7]:
one,two = env.reset()

In [8]:
print(type(one))
print(type(two))

<class 'numpy.ndarray'>
<class 'dict'>


In [10]:
print((one.shape))
print((two.keys()))

(210, 160, 3)
dict_keys(['lives', 'episode_frame_number', 'frame_number'])


In [8]:
for ep in dataset.episodes:
    print(type(ep.observations))
    print(type(ep))
    break

<class 'numpy.ndarray'>
<class 'd3rlpy.dataset.components.Episode'>
